In [1]:

from main.model.neegavi.factories.core import CoreFactory
from main.model.neegavi.utils import get_model_ckpt

path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/best-2-attn-1-beta/2026-03-27_22-46-58/checkpoints/epochepoch=39-stepstep=102120.ckpt"

ckpt = get_model_ckpt(weights_path=path)
backbone = CoreFactory.best_inference().build()
# Load state of the seed ckpt
backbone.load_state_dict(ckpt, strict=False)
backbone.eval()

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

EegInterAviModel(
  (pivot): ModalityStream(
    (adapter): EegAdapter(
      (ff): Sequential(
        (0): Linear(in_features=3800, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (supports): ModuleList(
    (0-1): 2 x ModalityStream(
      (adapter): PerceiverResamplerAdapter(
        (linear_reshape): Linear(in_features=768, out_features=384, bias=True)
        (resampler): PerceiverResampler(
          (blocks): ModuleList(
            (0-1): 2 x ModuleList(
              (0): PerceiverAttention(
                (norm_latents): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (norm_media): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (to_q): Linear(in_features=384, out_features=768, bias=False)
                (to_k): Linear(in_features=384, out_features=768, bias=False)
                (to_v): Linear(in_features=384, out_features=

In [2]:
from main.dataset.to_eegavi_processed import EegaviPersistentForward

fwd = EegaviPersistentForward(backbone, "/home/jacopo/dataset/PLAY")

In [3]:
from tensordict import TensorDict

local_td = TensorDict.load_memmap(
    "/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream/8400")

In [5]:
import torch


def dequantize(x: TensorDict, dtype=torch.float16):
    output: dict = {}
    for key, td in x.items():
        if "scales" in td and "data" in td:
            data = td["data"].to(dtype=dtype, non_blocking=True)
            data.mul_(td["scales"])  # For optimization reasons (I dislike it)

            td = {"data": data, "mask": td["mask"]}

        output[key] = td

    return TensorDict.from_dict(output)

In [6]:
local_td = dequantize(local_td)
local_td

TensorDict(
    fields={
        assessment: TensorDict(
            fields={
                labels: NonTensorStack(
                    [['arousal', 'valence', 'dominance', 'liking', 'fa...,
                    batch_size=torch.Size([7, 12]),
                    device=cpu),
                scales: NonTensorStack(
                    [[[[0.0, 9.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0],...,
                    batch_size=torch.Size([7, 1, 12, 2]),
                    device=cpu),
                scores: MemoryMappedTensor(shape=torch.Size([7, 12]), device=cpu, dtype=torch.float64, is_shared=True)},
            batch_size=torch.Size([7]),
            device=cpu,
            is_shared=False),
        aud: TensorDict(
            fields={
                data: Tensor(shape=torch.Size([7, 8, 199, 768]), device=cpu, dtype=torch.float16, is_shared=False),
                mask: MemoryMappedTensor(shape=torch.Size([7, 8]), device=cpu, dtype=torch.bool, is_shared=True)},
            batch_size=t

In [8]:
with torch.autocast(device_type="cpu", dtype=torch.float16):
    res = fwd(local_td)

In [9]:
res

TensorDict(
    fields={
        assessment: TensorDict(
            fields={
                labels: NonTensorStack(
                    [['arousal', 'valence', 'dominance', 'liking', 'fa...,
                    batch_size=torch.Size([7, 12]),
                    device=cpu),
                scales: NonTensorStack(
                    [[[[0.0, 9.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0],...,
                    batch_size=torch.Size([7, 1, 12, 2]),
                    device=cpu),
                scores: MemoryMappedTensor(shape=torch.Size([7, 12]), device=cpu, dtype=torch.float64, is_shared=True)},
            batch_size=torch.Size([7]),
            device=cpu,
            is_shared=False),
        eeg: TensorDict(
            fields={
                data: Tensor(shape=torch.Size([7, 32, 19, 200]), device=cpu, dtype=torch.float16, is_shared=False),
                mask: MemoryMappedTensor(shape=torch.Size([7, 32, 19]), device=cpu, dtype=torch.bool, is_shared=True)},
            batch_s

In [ ]:
# [T, t, C, 200] -> mean over t and C
# [T, 200] -> mean over T
# [200]

Applica questa strategia, permette di avere i dati più compatti e francamente lo trovo più pratico. <br>
Batch size può anche aumetare facilmente.